In [ ]:
""" Shotaro Ishihara """
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# モデルとトークナイザーの読み込み
model_id = "llm-jp/llm-jp-3-150m-instruct3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# pad_token_idの明示的な設定
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# プロンプトの設定
prompt = "The movie was full of"

# デコーディング方法と温度パラメータの設定
decoding_configs = [
    {"method": "greedy", "temperature": 0.1},
    {"method": "greedy", "temperature": 1.0},
    {"method": "greedy", "temperature": 2.0},
    {"method": "beam_search", "num_beams": 5, "temperature": 1.0},
    {"method": "top_k", "top_k": 50, "temperature": 1.0},
    {"method": "top_p", "top_p": 0.9, "temperature": 1.0},
]

# 各設定でテキスト生成を実行
for config in decoding_configs:
    print(f"\nデコーディング方法: {config['method']}")
    if config['method'] == "greedy":
        print(f"温度パラメータ: {config['temperature']}")

    # 入力のトークン化
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # 生成パラメータの設定
    gen_kwargs = {
        "max_new_tokens": 20,
        "pad_token_id": tokenizer.pad_token_id,
    }

    # デコーディング方法に応じたパラメータを追加
    if config["method"] == "greedy": # 1個の候補を保持しながら次トークンを探索
        gen_kwargs.update(
            {
                "temperature": config["temperature"],
                "do_sample": True,
            }
        )
    elif config["method"] == "beam_search": # 5個の候補を保持しながら次トークンを探索
        gen_kwargs.update(
            {
                "num_beams": config["num_beams"],
                "do_sample": True,
            }
        )
    elif config["method"] == "top_k": # 50個の候補から次トークンをランダムサンプリング
        gen_kwargs.update(
            {
                "top_k": config["top_k"],
                "temperature": config["temperature"],
                "do_sample": True,
                "top_p": 1.0,
            }
        )
    elif config["method"] == "top_p": # 確率が0.9を超える上位トークンから次トークンをランダムサンプリング
        gen_kwargs.update(
            {
                "top_p": config["top_p"],
                "temperature": config["temperature"],
                "do_sample": True,
                "top_k": 0,
            }
        )

    # テキスト生成
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids, 
            attention_mask=attention_mask, 
            **gen_kwargs
        )

    # 結果の表示
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"生成されたテキスト: {generated_text}")
    print("-" * 50)


デコーディング方法: greedy
温度パラメータ: 0.1
生成されたテキスト: The movie was full of memorable moments, including the opening scene of the movie, the opening of the movie, and the opening
--------------------------------------------------

デコーディング方法: greedy
温度パラメータ: 1.0
生成されたテキスト: The movie was full of hearty twists and turns, as each character' struggle to connect with their audience and their choices
--------------------------------------------------

デコーディング方法: greedy
温度パラメータ: 2.0
生成されたテキスト: The movie was full of pundits who believed everything seemed to be a myth. To argue over which movie took place and
--------------------------------------------------

デコーディング方法: beam_search
生成されたテキスト: The movie was full of twists and turns, and it had a profound impact on the audience's perception of the story.
--------------------------------------------------

デコーディング方法: top_k
生成されたテキスト: The movie was full of tension and controversy. The movie's story is just one aspect of the story that gets talked about.
-----